In [1]:
import pandas as pd
import numpy as np
import re

from scipy.interpolate import interp1d, PchipInterpolator
from sklearn.metrics import mean_squared_error

In [3]:
df = pd.read_csv("../data/raw/dataset.csv")

print(df.shape)
df.head()

(975, 30)


,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.12330,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.15760,0.15240,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150
1,07-01-2026 09:20,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,NaN,0.15420,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353
2,07-01-2026 09:25,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,0.15927,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN
3,07-01-2026 09:30,26128.95,0.10860,0.10842,0.11150,0.12248,0.10715,0.11098,0.10345,NaN,...,0.15755,NaN,0.14691,0.14209,0.13721,0.13184,0.12722,0.12252,0.11729,0.11200
4,07-01-2026 09:35,26131.90,0.10462,0.10538,0.12459,0.12051,0.11225,0.11294,0.10544,NaN,...,0.15924,0.15334,0.14784,0.14230,NaN,0.13219,0.12733,0.12295,0.11707,NaN


In [4]:
def get_strike(col):
    match = re.search(r'(\d+)(CE|PE)$', col)
    return int(match.group(1))

ce_cols = sorted(
    [col for col in df.columns if col.endswith("CE")],
    key=get_strike
)

pe_cols = sorted(
    [col for col in df.columns if col.endswith("PE")],
    key=get_strike
)

print("CE Columns:", len(ce_cols))
print("PE Columns:", len(pe_cols))

CE Columns: 14
PE Columns: 14


In [5]:
np.random.seed(42)

df_masked = df.copy()

mask_fraction = 0.10

true_values = []
locations = []

for row_idx in range(len(df_masked)):

    # CE columns
    available_ce = (
        df_masked.loc[row_idx, ce_cols]
        .dropna()
        .index
        .tolist()
    )

    if len(available_ce) >= 5:

        n_mask = max(
            1,
            int(len(available_ce) * mask_fraction)
        )

        cols_to_mask = np.random.choice(
            available_ce,
            size=n_mask,
            replace=False
        )

        for col in cols_to_mask:

            true_values.append(
                df_masked.at[row_idx, col]
            )

            locations.append(
                (row_idx, col)
            )

            df_masked.at[row_idx, col] = np.nan

    # PE columns
    available_pe = (
        df_masked.loc[row_idx, pe_cols]
        .dropna()
        .index
        .tolist()
    )

    if len(available_pe) >= 5:

        n_mask = max(
            1,
            int(len(available_pe) * mask_fraction)
        )

        cols_to_mask = np.random.choice(
            available_pe,
            size=n_mask,
            replace=False
        )

        for col in cols_to_mask:

            true_values.append(
                df_masked.at[row_idx, col]
            )

            locations.append(
                (row_idx, col)
            )

            df_masked.at[row_idx, col] = np.nan

true_values = np.array(true_values)

print("Validation Samples =", len(true_values))

Validation Samples = 1950


In [6]:
linear_df = df_masked.copy()

x_ce = np.arange(len(ce_cols))
x_pe = np.arange(len(pe_cols))

# CE
for row_idx in range(len(linear_df)):

    row = linear_df.loc[row_idx, ce_cols].values.astype(float)

    valid = ~np.isnan(row)

    if valid.sum() >= 2:

        f = interp1d(
            x_ce[valid],
            row[valid],
            kind="linear",
            fill_value="extrapolate"
        )

        missing = np.isnan(row)

        row[missing] = f(x_ce[missing])

        linear_df.loc[row_idx, ce_cols] = row

# PE
for row_idx in range(len(linear_df)):

    row = linear_df.loc[row_idx, pe_cols].values.astype(float)

    valid = ~np.isnan(row)

    if valid.sum() >= 2:

        f = interp1d(
            x_pe[valid],
            row[valid],
            kind="linear",
            fill_value="extrapolate"
        )

        missing = np.isnan(row)

        row[missing] = f(x_pe[missing])

        linear_df.loc[row_idx, pe_cols] = row

In [7]:
linear_preds = np.array([
    linear_df.at[row, col]
    for row, col in locations
])

linear_mse = mean_squared_error(
    true_values,
    linear_preds
)

print("Linear Interpolation MSE =", linear_mse)

Linear Interpolation MSE = 6.394331934739985e-05


In [8]:
pchip_df = df_masked.copy()

x_ce = np.arange(len(ce_cols))
x_pe = np.arange(len(pe_cols))

# CE
for row_idx in range(len(pchip_df)):

    row = pchip_df.loc[row_idx, ce_cols].values.astype(float)

    valid = ~np.isnan(row)

    if valid.sum() >= 2:

        f = PchipInterpolator(
            x_ce[valid],
            row[valid],
            extrapolate=True
        )

        missing = np.isnan(row)

        row[missing] = f(x_ce[missing])

        pchip_df.loc[row_idx, ce_cols] = row

# PE
for row_idx in range(len(pchip_df)):

    row = pchip_df.loc[row_idx, pe_cols].values.astype(float)

    valid = ~np.isnan(row)

    if valid.sum() >= 2:

        f = PchipInterpolator(
            x_pe[valid],
            row[valid],
            extrapolate=True
        )

        missing = np.isnan(row)

        row[missing] = f(x_pe[missing])

        pchip_df.loc[row_idx, pe_cols] = row

In [9]:
pchip_preds = np.array([
    pchip_df.at[row, col]
    for row, col in locations
])

pchip_mse = mean_squared_error(
    true_values,
    pchip_preds
)

print("PCHIP Interpolation MSE =", pchip_mse)

PCHIP Interpolation MSE = 8.16132081897085e-05
